# 1. Configuración, Descarga y Carga del Dataset (Kaggle API)
En esta sección nos conectamos de manera programática a la API de Kaggle para obtener el conjunto de datos de clasificación de calidad de frutas asegurando la reproducibilidad del proyecto.

In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 0. Cargar las variables del archivo .env ubicado en la raíz del proyecto
# (Asumiendo que el notebook está en 'notebooks/', subimos un nivel con '../.env')
from dotenv import load_dotenv, find_dotenv

# 1. Busca automáticamente el archivo .env buscando hacia arriba desde tu ubicación actual
load_dotenv(find_dotenv())

# 2. Inyectar el token en el entorno de Kaggle de forma segura
api_token = os.getenv("KAGGLE_API_TOKEN")
if api_token:
    os.environ['KAGGLE_API_TOKEN'] = api_token
else:
    print("Advertencia: No se encontró la variable KAGGLE_API_TOKEN en el archivo .env")

# 3. Autenticación con la API de Kaggle
try:
    import kaggle
    kaggle.api.authenticate()
    print("¡Autenticación exitosa con la API de Kaggle!")
except Exception as e:
    print(f"Error en la autenticación: {e}")

¡Autenticación exitosa con la API de Kaggle!


In [9]:
# Definir rutas relativas
BASE_DIR = Path().resolve().parent
KAGGLE_RAW_DIR = BASE_DIR / "data" / "raw" / "kaggle"
KAGGLE_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Descargar solo si la carpeta está vacía
if not any(KAGGLE_RAW_DIR.iterdir()):
    print("Descargando el dataset 'ryandpark/fruit-quality-classification'...")
    try:
        kaggle.api.dataset_download_files(
            'ryandpark/fruit-quality-classification', 
            path=str(KAGGLE_RAW_DIR), 
            unzip=True
        )
        print(f"¡Dataset descargado con éxito en: {KAGGLE_RAW_DIR}")
    except Exception as e:
        print(f"Error al descargar el dataset: {e}")
else:
    print("El dataset ya se encuentra descargado localmente.")

Descargando el dataset 'ryandpark/fruit-quality-classification'...
Dataset URL: https://www.kaggle.com/datasets/ryandpark/fruit-quality-classification
¡Dataset descargado con éxito en: C:\Users\sgall\Desktop\ProyectoAPOIII\src\data\raw\kaggle


# 2. Preprocesamiento y Normalización de Imágenes
Redimensionaremos todas las imágenes a un tamaño uniforme ($224 \times 224$ píxeles) para facilitar su posterior uso en modelos de Machine Learning y Deep Learning, organizándolas en la carpeta `data/processed/`.

In [12]:
import cv2

PROCESSED_DIR = BASE_DIR / "data" / "processed"
IMG_SIZE = (224, 224)

print("Iniciando preprocesamiento y redimensionamiento...")
processed_count = 0

if KAGGLE_RAW_DIR.exists():
    for class_dir in KAGGLE_RAW_DIR.iterdir():
        if class_dir.is_dir():
            class_name = class_dir.name
            print(f"  └─ Procesando categoría: {class_name}")
            
            # Usamos rglob("*.*") para buscar en todas las subcarpetas internas
            for img_file in class_dir.rglob("*.*"):
                if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    try:
                        img = cv2.imread(str(img_file))
                        if img is None:
                            continue
                        
                        # Redimensionar a 224x224
                        img_resized = cv2.resize(img, IMG_SIZE)
                        
                        # Crear un nombre único combinando la subruta para evitar que se sobrescriban
                        relative_path = img_file.relative_to(class_dir)
                        flat_filename = str(relative_path).replace("/", "_").replace("\\", "_")
                        
                        # Guardar en data/processed/
                        dest_path = PROCESSED_DIR / class_name / f"kaggle_{flat_filename}"
                        dest_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(dest_path), img_resized)
                        processed_count += 1
                    except Exception as ex:
                        print(f"Error procesando {img_file.name}: {ex}")

print(f"\n Proceso finalizado! Total de imágenes procesadas: {processed_count}")

Iniciando preprocesamiento y redimensionamiento...
  └─ Procesando categoría: Bad Quality_Fruits
  └─ Procesando categoría: Good Quality_Fruits
  └─ Procesando categoría: Mixed Qualit_Fruits

 Proceso finalizado! Total de imágenes procesadas: 19526


# 3. Verificación rápida de clases (EDA básico)
Contamos cuántas imágenes quedaron listas por categoría para detectar posibles desbalanceos en los datos.

In [13]:
import pandas as pd

data_summary = []
if PROCESSED_DIR.exists():
    for class_dir in PROCESSED_DIR.iterdir():
        if class_dir.is_dir():
            num_imgs = len(list(class_dir.glob("*.*")))
            data_summary.append({"Categoría": class_dir.name, "Cantidad de Imágenes": num_imgs})

df_summary = pd.DataFrame(data_summary)
print("Resumen de distribución de clases:")
display(df_summary)

Resumen de distribución de clases:


,Categoría,Cantidad de Imágenes
0,Bad Quality_Fruits,6788
1,Good Quality_Fruits,11664
2,Mixed Qualit_Fruits,1074
